#### 🚀 Transfer Learning

**Transfer Learning** is a powerful technique where you reuse a **pre-trained deep neural network (DNN)** instead of training a model from scratch.

Training from scratch is often:
- **Expensive**
- **Data-intensive**
- **Time-consuming**

For example, in **satellite imagery tasks**, leveraging an existing pre-trained model (like one trained on ImageNet) and reusing its lower layers can significantly speed up development and improve performance.

---

#### 🔑 Why It Works

Lower layers of neural networks learn **generic features** such as:
- Edges
- Textures
- Shapes

These features are often **transferable across different tasks**, especially when the domains are similar.

---

#### ⚙️ Key Considerations

- **Match input size**:  
  Resize your images to match the input size expected by the original model.

- **Feature similarity matters**:  
  Transfer learning works best when your dataset shares **low-level feature similarities** with the original training data.

- **Only useful in DNNs**:

  It turns out that transfer learning does not work very well with small dense networks, presumably because small networks learn few patterns, and
  dense networks learn very specific patterns, which are unlikely to be useful in other tasks

---

#### 🔍 How Many Layers to Reuse?

It depends on how similar your task is to the original one:

- **Very similar tasks**:
  - Reuse **most or all hidden layers**
  - Replace only the **output layer**

- **Less similar tasks**:
  - Reuse fewer layers
  - Train more layers from scratch

---

#### 🧪 Practical Strategy (SOP)

1. **Start by freezing all pre-trained layers**
   - Train only the new output layer
   - Evaluate performance

2. **Gradually unfreeze layers**
   - Unfreeze top 1–2 layers first
   - Fine-tune the network step by step

3. **Adjust learning rate**
   - Use a **lower learning rate** when unfreezing layers  
   - Prevents destroying useful pre-trained features

---


![Transfer Learning](Images/Transfer_Learning.png)

#### Practical implementation using keras 
ex-1 say the tasks are very similar 
Notice that the new layer in model_B_on_A is initiaized randomly. So this will perform bad in the starting for some epochs as it has no context of what was happening before. There are chances of large error gradients here. To avoid this, freeze the reused layers for some epochs, this gives the new layer(s) to get some context. To do this set the every layer's trainable attribute to False and compile the model 

In [ ]:
model_A = keras.models.load_model('path_to_model_A')
model_B_on_A = keras.models.Sequential(model_A.layers[:-1])
model_B_on_A.add(keras.layers.Dense(1, activation='softmax')) #Note - The output layer is initialized with random weights 

'''
Train the model for few epochs like this, and then unfreeze the layers one by one and train again. This is done to avoid large 
error gradients as the new layer is initialized randomly. 
'''
for layer in model_B_on_A[:-1]:
    layer.trainable = False

In [ ]:
history = model_B_on_A.fit(X_train_B, y_train_B, epochs=4, validation_data=(X_valid_B, y_valid_B))

for layer in model_B_on_A.layers[:-1]:
    layer.trainable = True
    
optimizer = keras.optimizers.SGD(learning_rate=1e-4)
model_B_on_A.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
history = model_B_on_A.fit(X_train_B, y_train_B, epochs=16, validation_data=(X_valid_B, y_valid_B))

#### Unsupervised Pre Training


#### 🤖 Leveraging Unlabeled Data with Unsupervised Pretraining

When you have **limited labeled data** but **abundant unlabeled data**, you can use **unsupervised learning** to pretrain your model and then adapt it for your specific task.

Common approaches include training:
- **Autoencoders**
- **GANs (Generative Adversarial Networks)**

The idea is to **learn useful feature representations** from unlabeled data, and then reuse these learned features for a supervised task.

---

#### 💡 When Should You Use This?

Use this approach when:
- You **don’t have a pre-trained model** available
- You have **very little labeled data**
- You have **plenty of unlabeled data**

---

#### ⚙️ Intuition

Unsupervised models learn:
- Structure in the data
- Important patterns and representations

These learned **low-level features** can then be reused, similar to transfer learning.

---

#### 🛠️ Procedure (Layer-wise Pretraining)

1. **Train the model on unlabeled data (layer by layer)**  
   - Start by training the first layer
   - Freeze it once trained  
   - Move to the next layer and repeat

2. **Gradually build the network**  
   - Each layer learns progressively higher-level representations
   - Previously trained layers remain **frozen**

3. **Add a task-specific output layer**  
   - Attach a new output (classification/regression head)

4. **Fine-tune with labeled data**  
   - Train the final layer using labeled data  
   - Optionally unfreeze upper layers for further tuning

---

#### 🔍 Why This Works

- The model learns **useful representations without labels**
- Reduces dependence on large labeled datasets
- Helps in better **initialization** compared to random weights

---



![Unsupervised Pre Training](Images/Unsupervised_pre_training.png)

#### 🔁 Pretraining on an Auxiliary Task

When you have **limited labeled data**, another powerful strategy is to **pretrain a neural network on an auxiliary task** where labeled data is easy to obtain or generate.

Once trained, you can **reuse the lower layers** of this network for your actual task and fine-tune it using your available labeled data.

---

#### 💡 Key Idea

Instead of relying only on scarce labeled data:
- Create or use a **related task with abundant data**
- Train a model to learn **useful representations**
- Transfer this knowledge to your main task

---

#### 🧠 Example in NLP

In **Natural Language Processing (NLP)**, large text corpora are easily available.

A common approach:
- Take millions of text documents
- **Mask random words in sentences**
- Train the model to **predict the missing words**

Example:
- Input: *“What ___ you saying?”*
- Target: **“are”** or **“were”**

This is similar to how models like **BERT** are trained.

---

#### 🚀 Why This Works

- The model learns:
  - Grammar
  - Context
  - Semantic relationships

- After pretraining:
  - It already has a strong understanding of language
  - Requires **less labeled data** for the final task

---

#### ⚙️ Workflow

1. **Choose an auxiliary task**
   - Should be related to your main problem
   - Should have **plenty of labeled or auto-generated data**

2. **Pretrain the model**
   - Train on the auxiliary task until good performance

3. **Reuse learned layers**
   - Transfer lower (and sometimes middle) layers

4. **Add task-specific head**
   - Replace output layer based on your task

5. **Fine-tune on labeled data**
   - Use a **lower learning rate**
   - Optionally unfreeze more layers gradually

---

#### 📈 Where This is Useful

- NLP (language modeling, masked prediction)
- Computer Vision (self-supervised tasks like rotation prediction)
- Speech and audio processing
- Any domain with **abundant raw data but limited labels**
